In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os

# ۱. تنظیمات مسیرها و متغیرها
file_path = r'second_stage_inputs\G12\dsas_g12_bearings_vibration_temp_output.xlsx'
output_filename1 = r'outputs\G12\dsas_g12_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g12_bearings_vibration_temp_correlation_detection_output1.xlsx'
output_filename2 = r'outputs\G12\dsas_g12_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g12_bearings_vibration_temp_correlation_detection_output2.xlsx'

os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

all_features = [
    'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
    'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
    'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
]

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# ۲. خواندن فایل اصلی
try:
    df_raw = pd.read_excel(file_path)
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values(by='date')
    print("فایل ورودی با موفقیت خوانده شد.")
except Exception as e:
    print(f"خطا در خواندن فایل: {e}")
    exit()

# ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

def calculate_distance(i):
    label, point = labels[i], scaled_data[i].reshape(1, -1)
    if label != -1:
        return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# ۴. بازه ۳۰ روز آخر
last_date = df_cleaned.index.max()
start_analysis_date = last_date - pd.Timedelta(days=30)
print(f"شروع تحلیل Rolling از تاریخ: {start_analysis_date}")

# ۵. محاسبه Rolling Correlation و ذخیره در لیست‌ها
results_list = [] # برای خروجی شماره ۱

print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# محاسبه کوریلیشن برای تمامی جفت‌ها به صورت یکجا برای بهینه‌سازی
for target in target_sensors:
    # ایجاد یک دیکشنری برای ذخیره مقادیر این تارگت در هر لحظه (برای خروجی ۲)
    # این دیکشنری در نهایت به دیتافریم تبدیل می‌شود
    temp_storage = {}

    for feature in all_features:
        if target == feature:
            # کوریلیشن با خودش را NaN در نظر می‌گیریم
            temp_storage[feature] = np.nan
            continue

        # محاسبه Rolling Correlation
        rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
        # فقط بازه ۳۰ روز آخر را نگه می‌داریم
        temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]


    # ساخت ردیف‌های خروجی ۲ برای این تارگت خاص
    target_df_wide = pd.DataFrame(temp_storage)
    target_df_wide['AssetID'] = target
    target_df_wide = target_df_wide.reset_index()

    # مرتب‌سازی ستون‌ها: date, AssetID و سپس بقیه
    cols = ['date', 'AssetID'] + all_features
    results_list.append(target_df_wide[cols])

# ۶. ترکیب نتایج و ساخت فایل‌های خروجی
df_output2 = pd.concat(results_list, ignore_index=True)

# تولید خروجی شماره ۱ از روی خروجی شماره ۲ (تبدیل Wide به Long)
# این کار باعث می‌شود محاسبات دوباره تکرار نشود و هر دو فایل با هم منطبق باشند
df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
                             value_vars=all_features, 
                             var_name='AssetID_correlation', 
                             value_name='correlation_value')

# حذف مقادیر خودش با خودش (NaN) در خروجی ۱
df_output1 = df_output1.dropna(subset=['correlation_value'])

# ۷. ذخیره نهایی
try:
    # ذخیره خروجی ۱ (ساختار ردیفی)
    df_output1.to_excel(output_filename1, index=False)
    # ذخیره خروجی ۲ (ساختار ستونی - درخواستی جدید)
    df_output2.to_excel(output_filename2, index=False)

    print("\nعملیات با موفقیت پایان یافت.")
    print(f"فایل ۱ (ساختار ردیفی): {output_filename1}")
    print(f"فایل ۲ (ساختار ستونی): {output_filename2}")
    print(f"تعداد ردیف‌های خروجی نهایی: {len(df_output2)}")
except Exception as e:
    print(f"خطا در ذخیره فایل‌ها: {e}")

# نمایش نمونه خروجی ۲
print("\nنمونه ساختار فایل شماره ۲:")
print(df_output2.head())

فایل ورودی با موفقیت خوانده شد.
شروع تحلیل Rolling از تاریخ: 2023-11-02 12:49:12
در حال محاسبه ماتریس کوریلیشن متحرک...

عملیات با موفقیت پایان یافت.
فایل ۱ (ساختار ردیفی): outputs\G12\dsas_g12_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g12_bearings_vibration_temp_correlation_detection_output1.xlsx
فایل ۲ (ساختار ستونی): outputs\G12\dsas_g12_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g12_bearings_vibration_temp_correlation_detection_output2.xlsx
تعداد ردیف‌های خروجی نهایی: 680

نمونه ساختار فایل شماره ۲:
                 date       AssetID  AssetID_9358  AssetID_9359  AssetID_9360  \
0 2023-11-02 14:00:48  AssetID_9358           NaN      0.723959      0.147100   
1 2023-11-02 16:34:35  AssetID_9358           NaN      0.721651      0.141168   
2 2023-11-02 19:35:53  AssetID_9358           NaN      0.721996      0.142209   
3 2023-11-02 22:32:42  AssetID_9358           NaN      0.721878      0.144007   
4 2023-11-03 01:32:53  As